# Dark Chat — Fine-tuning Mistral 7B (LoRA + FP16)

**Модель:** Mistral 7B Instruct v0.3

**Техники:** LoRA + Gradient Checkpointing + FP16

**Время:** ~3-4 часа на T4 GPU

In [ ]:
!pip install -q transformers datasets trl peft accelerate

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'Память: {props.total_memory / 1024**3:.1f} GB')

In [ ]:
training_data = [
    {"instruction": "Привет", "output": "Привет! Чем могу помочь?"},
    {"instruction": "Кто ты?", "output": "Я Dark Chat — AI-ассистент. Могу отвечать на вопросы, генерировать код и помогать с задачами."},
    {"instruction": "Что ты умеешь?", "output": "Я умею: отвечать на вопросы, писать код на разных языках, объяснять сложные вещи простыми словами."},
    {"instruction": "Напиши hello world на Python", "output": "print('Hello, World!')"},
    {"instruction": "Напиши hello world на JavaScript", "output": "console.log('Hello, World!');"},
    {"instruction": "Напиши hello world на Go", "output": "package main\n\nimport \"fmt\"\n\nfunc main() {\n    fmt.Println(\"Hello, World!\")\n}"},
    {"instruction": "Что такое переменная?", "output": "Переменная — это именованная область памяти, в которой хранится значение. Например: x = 5."},
    {"instruction": "Объясни интернет простыми словами", "output": "Интернет — это глобальная сеть компьютеров, соединённых друг с другом."},
    {"instruction": "2+2=", "output": "4"},
    {"instruction": "Что такое AI?", "output": "AI — искусственный интеллект. Программы, которые могут учиться и принимать решения, как человек."},
    {"instruction": "Напиши функцию факториала на Python", "output": "def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)"},
    {"instruction": "Что такое массив?", "output": "Массив — структура данных, хранящая список элементов. Доступ по индексу: arr[0]."},
    {"instruction": "Расскажи анекдот", "output": "Программист — это тот, кто решает проблему, о которой ты не знал, способом, который ты не понимаешь."},
    {"instruction": "Как создать сайт?", "output": "Нужно: HTML (структура), CSS (стили), JavaScript (логика). Фреймворки: React, Vue, Django."},
    {"instruction": "Что такое API?", "output": "API — интерфейс для общения программ. Набор правил, по которым программы общаются друг с другом."},
    {"instruction": "Как работает Python?", "output": "Python — интерпретируемый язык. Код читается построчно и выполняется виртуальной машиной CPython."},
    {"instruction": "Что такое база данных?", "output": "База данных — организованное хранилище информации. Примеры: SQLite, PostgreSQL, MySQL."},
    {"instruction": "Напиши сортировку пузырьком", "output": "def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n-i-1):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n    return arr"},
    {"instruction": "Что такое Git?", "output": "Git — система контроля версий. Отслеживает изменения, помогает работать в команде."},
    {"instruction": "Объясни Docker", "output": "Docker — контейнеризация. Упакует приложение с зависимостями, работает одинаково везде."},
    {"instruction": "Что такое REST API?", "output": "REST API — стиль веб-API. Методы: GET, POST, PUT, DELETE."},
    {"instruction": "Напиши чтение файла на Python", "output": "with open('file.txt', 'r') as f:\n    content = f.read()\nprint(content)"},
    {"instruction": "Что такое рекурсия?", "output": "Рекурсия — функция вызывает сама себя. Нужно условие остановки."},
    {"instruction": "Как создать API на Python?", "output": "from fastapi import FastAPI\napp = FastAPI()\n\n@app.get('/')\ndef home():\n    return {'message': 'Hello'}"},
    {"instruction": "Что такое SQL?", "output": "SQL — язык запросов. Команды: SELECT, INSERT, UPDATE, DELETE."},
    {"instruction": "Напиши чат-бот на Python", "output": "while True:\n    msg = input('Ты: ')\n    if msg == 'привет':\n        print('Бот: Привет!')\n    elif msg == 'пока':\n        break"},
    {"instruction": "Что такое ООП?", "output": "ООП — объектно-ориентированное программирование. Принципы: инкапсуляция, наследование, полиморфизм."},
    {"instruction": "Объясни async/await", "output": "async/await — асинхронное программирование. Позволяет выполнять задачи параллельно."},
    {"instruction": "Что такое middleware?", "output": "Middleware — промежуточный слой. Обрабатывает данные между запросом и ответом."},
    {"instruction": "Напиши HTTP-сервер", "output": "from http.server import HTTPServer, SimpleHTTPRequestHandler\nserver = HTTPServer(('localhost', 8000), SimpleHTTPRequestHandler)\nserver.serve_forever()"},
    {"instruction": "Что такое CI/CD?", "output": "CI — автоматическая сборка. CD — автоматический деплой. Инструменты: GitHub Actions."},
]
print(f'Датасет: {len(training_data)} примеров')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'Загружаю {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    low_cpu_mem_usage=True,
)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Память: {torch.cuda.memory_allocated() / 1024**3:.1f} GB')

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'v_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

def format_example(example):
    return {'text': f"### Инструкция:\n{example['instruction']}\n\n### Ответ:\n{example['output']}"}

dataset = Dataset.from_list([format_example(d) for d in training_data])

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
split = tokenized.train_test_split(test_size=0.1)

print(f'Train: {len(split["train"])}, Eval: {len(split["test"])}')

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./darkchat-mistral-lora',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,
    save_steps=200,
    fp16=True,
    optim='adamw_torch',
    report_to='none',
    eval_strategy='steps',
    eval_steps=50,
    save_total_limit=2,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('=== Техники ===')
print('1. LoRA (r=8, alpha=16)')
print('2. Gradient Checkpointing')
print('3. Mixed Precision (FP16)')
print('\nОбучение...')
trainer.train()
print('Готово!')

In [ ]:
trainer.save_model('./darkchat-mistral-lora')
tokenizer.save_pretrained('./darkchat-mistral-lora')
print('Модель сохранена!')

In [ ]:
model.eval()

def generate(prompt):
    inputs = tokenizer(f'### Инструкция:\n{prompt}\n\n### Ответ:\n', return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split('### Ответ:\n')[-1]

print('=== Тест Mistral 7B + LoRA ===')
print()
print('Q: Привет!')
print(f'A: {generate("Привет!")}')
print()
print('Q: Напиши hello world на Python')
print(f'A: {generate("Напиши hello world на Python")}')
print()
print('Q: Что такое API?')
print(f'A: {generate("Что такое API?")}')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('darkchat-mistral-lora', 'zip', './darkchat-mistral-lora')
files.download('darkchat-mistral-lora.zip')
print('Скачано!')